# TopicGPT Tuning: c-TF-IDF & Topic Filtering Optimization

This notebook tunes the c-TF-IDF and topic filtering parameters for TopicGPT's
assignment stage. It **reuses** existing assignment checkpoints from the modeling
phase and grid-searches over:

- **`min_docs`** — Minimum documents per topic to keep
- **`ngram_range`** — n-gram range for c-TF-IDF CountVectorizer: `(1,2)` or `(1,3)`
- **`max_features`** — Vocabulary size cap: `5000`, `10000`, `50000`, or `None`

Each subject uses its best sentence transformer model identified during modeling.

In [1]:
import os
import time
import gc
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import product
from tqdm import tqdm
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from sklearn.feature_extraction.text import TfidfVectorizer
from itertools import combinations
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Configuration

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]
VERSION = "v1"

BASE_DIR = Path("../../../../data/preprocess")
CHECKPOINT_DIR = Path("../../../../models/topicGpt")
RESULT_DIR = Path("../../../../results/topicGpt/tunning")
TUNING_CKPT_DIR = CHECKPOINT_DIR  # tuning checkpoints go alongside modeling ones
EMBEDDING_DIR = Path("../../../../embedding")

# Best embedding model per subject (from modeling phase)
BEST_MODEL_MAP = {
    "cs":      "all_mpnet_base_v2",
    "math":    "sentence_transformers_all_MiniLM_L6_v2",
    "physics":  "all_mpnet_base_v2",
}

# --- Tuning grid ---
MIN_DOCS_VALUES     = [10, 25, 50, 100, 150, 200, 250]
NGRAM_RANGE_VALUES  = [(1, 1)]
MAX_FEATURES_VALUES = [10000, 50000, None]

TOP_N_WORDS = 10
RBO_P = 0.9

# Create output dirs
for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Subjects: {LIST_SUBJECT}")
print(f"min_docs grid:      {MIN_DOCS_VALUES}")
print(f"ngram_range grid:   {NGRAM_RANGE_VALUES}")
print(f"max_features grid:  {MAX_FEATURES_VALUES}")
print(f"Total combos per subject: {len(MIN_DOCS_VALUES) * len(NGRAM_RANGE_VALUES) * len(MAX_FEATURES_VALUES)}")
print(f"Results dir: {RESULT_DIR.resolve()}")

Subjects: ['cs', 'math', 'physics']
min_docs grid:      [10, 25, 50, 100, 150, 200, 250]
ngram_range grid:   [(1, 1)]
max_features grid:  [10000, 50000, None]
Total combos per subject: 21
Results dir: /home/nedo/Kuliah/TA/Program/results/topicGpt/tunning


## Checkpoint Utilities

In [3]:
def save_checkpoint(data, name: str, subject: str):
    """Save checkpoint to disk."""
    path = TUNING_CKPT_DIR / subject / f"{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(data, f)
    print(f"  Checkpoint saved: {path}")

def load_checkpoint(name: str, subject: str):
    """Load checkpoint from disk, return None if not found."""
    path = TUNING_CKPT_DIR / subject / f"{name}.pkl"
    if path.exists():
        with open(path, "rb") as f:
            data = pickle.load(f)
        print(f"  Checkpoint loaded: {path}")
        return data
    return None

## Data Loading

In [4]:
def load_dataset(subject: str) -> pd.DataFrame:
    """Load preprocessed dataset."""
    file_path = BASE_DIR / subject / "emb" / f"{VERSION}.csv"
    df = pd.read_csv(file_path)
    return df

all_data = {}
for subject in LIST_SUBJECT:
    df = load_dataset(subject)
    all_data[subject] = df
    print(f"{subject}: {len(df):,} documents loaded")

print(f"\nAll subjects loaded.")

cs: 165,756 documents loaded
math: 157,085 documents loaded
physics: 146,311 documents loaded

All subjects loaded.


## Generate Base Assignments

Instead of loading filtered assignment checkpoints from the modeling phase, we dynamically recompute the base assignments using the best sentence transformer for each subject.
This ensures zero filtering (`min_docs=0`) before the tuning loop begins.

In [5]:
MODEL_HF_MAP = {
    "all_mpnet_base_v2": "sentence-transformers/all-mpnet-base-v2",
    "sentence_transformers_all_MiniLM_L6_v2": "sentence-transformers/all-MiniLM-L6-v2"
}

# def load_doc_embeddings(subject: str, model_name: str):
#     path = EMBEDDING_DIR / subject / "emb" / f"{model_name}.npy"
#     if path.exists():
#         return np.load(path)
#     return None
def load_doc_embeddings(subject: str, model_name: str) -> np.ndarray:
    meta_path = EMBEDDING_DIR / subject / f"{model_name}_meta_v1.npy"
    if not meta_path.exists():
        meta_path = EMBEDDING_DIR / subject / f"{model_name}_v1_meta.npy"
    meta  = np.load(meta_path, allow_pickle=True).item()
    n, d  = meta["n_samples"], meta["emb_dim"]
    return np.memmap(
        EMBEDDING_DIR / subject / f"{model_name}_v1.mmap",
        dtype="float32", mode="r", shape=(n, d)
    )

def assign_topics_st(doc_embs, enriched_topics: dict, model_name: str, batch_size=2000, threshold=0.5):
    """Cosine-similarity assignment using a SentenceTransformer model with outlier thresholding."""
    hf_name  = MODEL_HF_MAP.get(model_name, model_name)
    st_model = SentenceTransformer(hf_name)
    
    topic_ids  = sorted(enriched_topics.keys())
    descs      = [enriched_topics[t].get("enriched_description") or enriched_topics[t]["description"]
                  for t in topic_ids]
    
    topic_embs = st_model.encode(descs, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    
    rows = []
    for start in tqdm(range(0, len(doc_embs), batch_size), desc=f"Assigning ({model_name})", leave=False):
        batch = np.array(doc_embs[start:start+batch_size])
        
        sims  = cos_sim(batch, topic_embs)
        best  = sims.argmax(axis=1)
        scores = sims.max(axis=1)
        
        for idx, (bi, sc) in enumerate(zip(best, scores)):
            if sc >= threshold:
                tid = topic_ids[bi]
                label = enriched_topics[tid]["label"]
            else:
                tid = -1
                label = "Outlier / Unassigned"
                
            rows.append({
                "doc_idx": start + idx, 
                "topic_id": tid,
                "topic_label": label,
                "confidence": float(sc)
            })
    return pd.DataFrame(rows)

all_base_assignments = {}

for subject in LIST_SUBJECT:
    best_model = BEST_MODEL_MAP[subject]
    
    # 1. Load document embeddings
    doc_embs = load_doc_embeddings(subject, best_model)
    if doc_embs is None:
        print(f"  ERROR: No doc embeddings found for {subject}/{best_model}")
        continue
        
    # 2. Load enriched topics
    enriched_ckpt = load_checkpoint("enrichment", subject)
    if not enriched_ckpt:
        print(f"  ERROR: No enriched topics found for {subject}")
        continue
        
    enriched_topics = enriched_ckpt["enriched_topics"]
    
    # 3. Assign topics
    print(f"Reassigning {subject} using {best_model}...")
    df_assign = assign_topics_st(doc_embs, enriched_topics, best_model)
    
    all_base_assignments[subject] = df_assign
    n_docs = len(df_assign)
    n_topics = df_assign["topic_id"].nunique()
    print(f"  {subject}: {n_docs:,} assignments, {n_topics} unique topics")

print(f"\nGenerated base assignments for {len(all_base_assignments)}/{len(LIST_SUBJECT)} subjects.")

  Checkpoint loaded: ../../../../models/topicGpt/cs/enrichment.pkl
Reassigning cs using all_mpnet_base_v2...


  cs: 165,756 assignments, 617 unique topics
  Checkpoint loaded: ../../../../models/topicGpt/math/enrichment.pkl
Reassigning math using sentence_transformers_all_MiniLM_L6_v2...


  math: 157,085 assignments, 611 unique topics
  Checkpoint loaded: ../../../../models/topicGpt/physics/enrichment.pkl
Reassigning physics using all_mpnet_base_v2...


  physics: 146,311 assignments, 505 unique topics

Generated base assignments for 3/3 subjects.


## Tuning Functions

In [6]:
def filter_and_reindex_topics(df_assign: pd.DataFrame, min_docs: int) -> pd.DataFrame:
    """Filter topics with fewer than min_docs documents and reindex."""
    df_filtered = df_assign[df_assign["topic_id"] != -1].copy()

    topic_counts = df_filtered["topic_id"].value_counts()
    valid_topics = topic_counts[topic_counts >= min_docs].index

    df_filtered = df_filtered[df_filtered["topic_id"].isin(valid_topics)].copy()

    unique_topics = sorted(df_filtered["topic_id"].unique())
    topic_mapping = {old_id: new_idx for new_idx, old_id in enumerate(unique_topics)}

    df_filtered["original_topic_id"] = df_filtered["topic_id"]
    df_filtered["topic_id"] = df_filtered["topic_id"].map(topic_mapping)

    return df_filtered



import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

def compute_topic_words_ctfidf(assignment_df: pd.DataFrame, texts: list,
                                top_n: int = 10,
                                ngram_range: tuple = (1, 2),
                                max_features: int = None) -> dict:
    """c-TF-IDF sesungguhnya sesuai formula BERTopic."""
    
    # 1. Kumpulkan dokumen per topik (tetap gabung, tapi hitung TF berbeda)
    topic_docs = {}
    for tid, grp in assignment_df.groupby("topic_id"):
        if tid == -1:
            continue
        combined = " ".join([texts[i] for i in grp["doc_idx"] if i < len(texts)])
        topic_docs[tid] = combined

    if not topic_docs:
        return {}

    tids = list(topic_docs.keys())
    docs = [topic_docs[t] for t in tids]

    # 2. Hitung count matrix (raw term frequency, bukan TF-IDF)
    vec = CountVectorizer(
        max_features=max_features,
        ngram_range=ngram_range
    )
    count_matrix = vec.fit_transform(docs).toarray().astype(float)
    feature_names = vec.get_feature_names_out()

    # 3. TF: frekuensi kata / total kata dalam topik tersebut
    words_per_topic = count_matrix.sum(axis=1, keepdims=True)
    tf = count_matrix / (words_per_topic + 1e-9)

    # 4. IDF ala c-TF-IDF: log(1 + A / tf_per_word)
    A = count_matrix.sum(axis=1).mean()          # rata-rata kata per topik
    tf_per_word = count_matrix.sum(axis=0)        # total frekuensi tiap kata
    idf = np.log(1 + A / (tf_per_word + 1e-9))   # shape: (n_features,)

    # 5. c-TF-IDF score
    ctfidf_matrix = tf * idf  # broadcasting otomatis

    # 6. Ambil top-N kata per topik
    topic_words = {}
    for i, tid in enumerate(tids):
        row = ctfidf_matrix[i]
        top_ids = row.argsort()[-top_n:][::-1]
        topic_words[tid] = [feature_names[idx] for idx in top_ids if row[idx] > 0]

    return topic_words

def rbo(list1, list2, p=0.9):
    if not list1 and not list2:
        return 1.0
    if not list1 or not list2:
        return 0.0

    # assign short (S) and long (L)
    if len(list1) <= len(list2):
        S, L = list1, list2
    else:
        S, L = list2, list1

    s, l = len(S), len(L)

    S_seen = set()
    L_seen = set()

    X = 0  # overlap
    rbo = 0.0
    disjoint = 0.0
    ext_term = 0.0

    for d in range(l):
        if d < s:
            s_item = S[d]
            S_seen.add(s_item)
        else:
            s_item = None

        l_item = L[d]
        L_seen.add(l_item)

        overlap_incr = 0

        if d < s:
            if s_item == l_item:
                overlap_incr = 1
            else:
                if s_item in L_seen:
                    overlap_incr += 1
                if l_item in S_seen:
                    overlap_incr += 1
        else:
            if l_item in S_seen:
                overlap_incr = 1

        X += overlap_incr

        if d < s:
            A_d = 2.0 * X / (len(S_seen) + len(L_seen))
        else:
            A_d = X / (d + 1)

        rbo += (1 - p) * (p ** d) * A_d

        if d < s:
            ext_term = A_d * (p ** (d + 1))
        else:
            X_s = X - overlap_incr if d == s else X_s
            disjoint += (1 - p) * (p ** d) * (
                X_s * (d + 1 - s) / ((d + 1) * s)
            )
            ext_term = (
                ((X - X_s) / (d + 1) + X_s / s)
                * (p ** (d + 1))
            )

        # optional optimization (safe)
        if p ** d < 1e-12:
            break

    return min(max(rbo + disjoint + ext_term, 0.0), 1.0)

def compute_coherence_irbo(assignment_df: pd.DataFrame, texts: list,
                           top_n: int = 10,
                           ngram_range: tuple = (1, 2),
                           max_features: int = None) -> dict:
    """Compute C_v coherence and IRBO diversity with parametric max_df."""
    topic_words = compute_topic_words_ctfidf(
        assignment_df, texts, top_n,
        ngram_range=ngram_range, max_features=max_features
    )
    word_lists  = [v for v in topic_words.values() if v]

    tokenized = [t.lower().split() for t in texts]

    # Flatten bigrams to unigrams for gensim
    gensim_word_lists = []
    for words in word_lists:
        topic_unigrams = []
        for w in words:
            topic_unigrams.extend(w.split())
        unique_unigrams = list(dict.fromkeys(topic_unigrams))[:top_n]
        gensim_word_lists.append(unique_unigrams)

    try:
        dct = Dictionary(tokenized)
        cm  = CoherenceModel(
            topics=gensim_word_lists,
            texts=tokenized,
            dictionary=dct,
            coherence="c_v",
            processes=5
        )
        cv = cm.get_coherence()
    except Exception as e:
        print(f"    Coherence error: {e}")
        cv = 0.0
    def irbo(lists1, lists2, p = 0.9):
        return 1 - rbo(lists1, lists2)



    pairs = list(combinations(word_lists, 2))
    irbo  = float(np.mean([irbo(a, b) for a, b in pairs])) if pairs else 0.0
    tq    = 2 * cv * irbo / (cv + irbo + 1e-8)

    return {"coherence": cv, "irbo": irbo, "topic_quality": tq}

## Topic Counts Recap

Display all topics with their document counts for all subjects before the tuning loop.

In [7]:
for subject, df_assign in all_base_assignments.items():
    print(f"=== {subject.upper()} ===")
    topic_counts = df_assign["topic_id"].value_counts().sort_values(ascending=False)
    display(pd.DataFrame({'topic': topic_counts.index, 'doc_count': topic_counts.values}))
    print("\n")

=== CS ===


,topic,doc_count
0,-1,62445
1,432,1728
2,481,1405
3,456,1302
4,586,1193
...,...,...
612,415,3
613,79,2
614,453,2
615,278,1




=== MATH ===


,topic,doc_count
0,-1,76778
1,519,1339
2,101,945
3,43,848
4,439,777
...,...,...
606,309,2
607,449,1
608,113,1
609,105,1




=== PHYSICS ===


,topic,doc_count
0,-1,55676
1,50,2543
2,80,2123
3,209,1879
4,406,1509
...,...,...
500,425,3
501,394,3
502,433,2
503,228,2


## Tuning Loop

Grid search over `min_docs × ngram_range × max_features` for each subject.
Reuses the base assignment DataFrames from modeling checkpoints.

In [8]:
all_tuning_results = []

for subject in LIST_SUBJECT:
    if subject not in all_base_assignments:
        print(f"\nSkipping {subject} (no base assignment)")
        continue

    print(f"\n{'='*60}")
    print(f"TUNING: {subject.upper()}")
    print(f"{'='*60}")

    base_df = all_base_assignments[subject]
    texts = all_data[subject]["text"].fillna("").tolist()
    best_model = BEST_MODEL_MAP[subject]

    combos = list(product(MIN_DOCS_VALUES, NGRAM_RANGE_VALUES, MAX_FEATURES_VALUES))
    best_tq, best_config = -1, None

    for min_docs, ngram_range, max_features in tqdm(combos, desc=f"Tuning {subject}"):
        ngram_str = f"{ngram_range[0]}{ngram_range[1]}"
        feat_str  = str(max_features) if max_features is not None else "none"
        ckpt_name = (
            f"tuning_{best_model}_mindocs{min_docs}"
            f"_ngram{ngram_str}_maxfeat{feat_str}"
        )

        # Check if already computed
        time_seconds = None
        ckpt = load_checkpoint(ckpt_name, subject)
        if ckpt:
            metrics      = ckpt["metrics"]
            n_topics     = ckpt["n_topics"]
            time_seconds = ckpt.get("time_seconds", None)
        else:
            # Apply filtering with current min_docs
            df_filtered = filter_and_reindex_topics(base_df, min_docs=min_docs)
            n_topics = df_filtered["topic_id"].nunique()

            if n_topics == 0:
                print(f"  min_docs={min_docs}, ngram={ngram_range}, max_feat={max_features}: 0 topics, skipping")
                metrics = {"coherence": 0.0, "irbo": 0.0, "topic_quality": 0.0}
                time_seconds = 0.0
            else:
                _t0 = time.time()
                metrics = compute_coherence_irbo(
                    df_filtered, texts,
                    ngram_range=ngram_range,
                    max_features=max_features
                )
                time_seconds = time.time() - _t0

            save_checkpoint(
                {"metrics": metrics, "n_topics": n_topics,
                 "min_docs": min_docs,
                 "ngram_range": ngram_range,
                 "max_features": max_features,
                 "time_seconds": time_seconds},
                ckpt_name, subject
            )

        row = {
            "subject":       subject,
            "best_model":    best_model,
            "min_docs":      min_docs,
            "ngram_range":   str(ngram_range),
            "max_features":  max_features,
            "n_topics":      n_topics,
            "coherence":     metrics["coherence"],
            "irbo":          metrics["irbo"],
            "topic_quality": metrics["topic_quality"],
            "time_seconds":  time_seconds,
        }
        all_tuning_results.append(row)

        if metrics["topic_quality"] > best_tq:
            best_tq = metrics["topic_quality"]
            best_config = row

        tqdm.write(
            f"  min_docs={min_docs:3d}  ngram={ngram_range}  max_feat={str(max_features):>5}  "
            f"topics={n_topics:3d}  C_v={metrics['coherence']:.4f}  "
            f"IRBO={metrics['irbo']:.4f}  TQ={metrics['topic_quality']:.4f}"
        )

    print(f"\n  BEST for {subject}: min_docs={best_config['min_docs']}, "
          f"ngram={best_config['ngram_range']}, max_feat={best_config['max_features']}, "
          f"TQ={best_tq:.4f}")

tuning_df = pd.DataFrame(all_tuning_results)
print(f"\nTotal results: {len(tuning_df)}")



TUNING: CS


Tuning cs: 100%|██████████| 21/21 [00:00<00:00, 773.30it/s]


  Checkpoint loaded: ../../../../models/topicGpt/cs/tuning_all_mpnet_base_v2_mindocs10_ngram11_maxfeat10000.pkl
  min_docs= 10  ngram=(1, 1)  max_feat=10000  topics=585  C_v=0.6384  IRBO=0.9942  TQ=0.7775
  Checkpoint loaded: ../../../../models/topicGpt/cs/tuning_all_mpnet_base_v2_mindocs10_ngram11_maxfeat50000.pkl
  min_docs= 10  ngram=(1, 1)  max_feat=50000  topics=585  C_v=0.6084  IRBO=0.9949  TQ=0.7551
  Checkpoint loaded: ../../../../models/topicGpt/cs/tuning_all_mpnet_base_v2_mindocs10_ngram11_maxfeatnone.pkl
  min_docs= 10  ngram=(1, 1)  max_feat= None  topics=585  C_v=0.6079  IRBO=0.9949  TQ=0.7547
  Checkpoint loaded: ../../../../models/topicGpt/cs/tuning_all_mpnet_base_v2_mindocs25_ngram11_maxfeat10000.pkl
  min_docs= 25  ngram=(1, 1)  max_feat=10000  topics=515  C_v=0.6608  IRBO=0.9934  TQ=0.7937
  Checkpoint loaded: ../../../../models/topicGpt/cs/tuning_all_mpnet_base_v2_mindocs25_ngram11_maxfeat50000.pkl
  min_docs= 25  ngram=(1, 1)  max_feat=50000  topics=515  C_v=0.6408 

Tuning math: 100%|██████████| 21/21 [00:00<00:00, 840.38it/s]


  Checkpoint loaded: ../../../../models/topicGpt/math/tuning_sentence_transformers_all_MiniLM_L6_v2_mindocs10_ngram11_maxfeat10000.pkl
  min_docs= 10  ngram=(1, 1)  max_feat=10000  topics=565  C_v=0.6003  IRBO=0.9944  TQ=0.7486
  Checkpoint loaded: ../../../../models/topicGpt/math/tuning_sentence_transformers_all_MiniLM_L6_v2_mindocs10_ngram11_maxfeat50000.pkl
  min_docs= 10  ngram=(1, 1)  max_feat=50000  topics=565  C_v=0.5815  IRBO=0.9948  TQ=0.7340
  Checkpoint loaded: ../../../../models/topicGpt/math/tuning_sentence_transformers_all_MiniLM_L6_v2_mindocs10_ngram11_maxfeatnone.pkl
  min_docs= 10  ngram=(1, 1)  max_feat= None  topics=565  C_v=0.5813  IRBO=0.9948  TQ=0.7338
  Checkpoint loaded: ../../../../models/topicGpt/math/tuning_sentence_transformers_all_MiniLM_L6_v2_mindocs25_ngram11_maxfeat10000.pkl
  min_docs= 25  ngram=(1, 1)  max_feat=10000  topics=497  C_v=0.6231  IRBO=0.9934  TQ=0.7658
  Checkpoint loaded: ../../../../models/topicGpt/math/tuning_sentence_transformers_all_Mi

Tuning physics: 100%|██████████| 21/21 [00:00<00:00, 1076.95it/s]

  Checkpoint loaded: ../../../../models/topicGpt/physics/tuning_all_mpnet_base_v2_mindocs10_ngram11_maxfeat10000.pkl
  min_docs= 10  ngram=(1, 1)  max_feat=10000  topics=479  C_v=0.6567  IRBO=0.9935  TQ=0.7908
  Checkpoint loaded: ../../../../models/topicGpt/physics/tuning_all_mpnet_base_v2_mindocs10_ngram11_maxfeat50000.pkl
  min_docs= 10  ngram=(1, 1)  max_feat=50000  topics=479  C_v=0.6406  IRBO=0.9940  TQ=0.7791
  Checkpoint loaded: ../../../../models/topicGpt/physics/tuning_all_mpnet_base_v2_mindocs10_ngram11_maxfeatnone.pkl
  min_docs= 10  ngram=(1, 1)  max_feat= None  topics=479  C_v=0.6404  IRBO=0.9940  TQ=0.7789
  Checkpoint loaded: ../../../../models/topicGpt/physics/tuning_all_mpnet_base_v2_mindocs25_ngram11_maxfeat10000.pkl
  min_docs= 25  ngram=(1, 1)  max_feat=10000  topics=427  C_v=0.6789  IRBO=0.9924  TQ=0.8062
  Checkpoint loaded: ../../../../models/topicGpt/physics/tuning_all_mpnet_base_v2_mindocs25_ngram11_maxfeat50000.pkl
  min_docs= 25  ngram=(1, 1)  max_feat=50000

## Results Summary

In [9]:
# Save full results CSV per subject
for subject in LIST_SUBJECT:
    subj_df = tuning_df[tuning_df["subject"] == subject]
    if len(subj_df) == 0:
        continue
    out_path = RESULT_DIR / subject / "tuning_results.csv"
    subj_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

# Also save combined results
combined_path = RESULT_DIR / "tuning_results_all.csv"
tuning_df.to_csv(combined_path, index=False)
print(f"\nCombined results saved: {combined_path}")

# Show best per subject
print(f"\n{'='*80}")
print(f"BEST CONFIGURATION PER SUBJECT")
print(f"{'='*80}")
for subject in LIST_SUBJECT:
    subj_df = tuning_df[tuning_df["subject"] == subject]
    if len(subj_df) == 0:
        continue
    best = subj_df.loc[subj_df["topic_quality"].idxmax()]
    print(f"\n{subject.upper()}:")
    print(f"  Model:        {best['best_model']}")
    print(f"  min_docs:     {int(best['min_docs'])}")
    print(f"  ngram_range:  {best['ngram_range']}")
    print(f"  max_features: {best['max_features']}")
    print(f"  Topics:       {int(best['n_topics'])}")
    print(f"  C_v:          {best['coherence']:.4f}")
    print(f"  IRBO:         {best['irbo']:.4f}")
    print(f"  TQ:           {best['topic_quality']:.4f}")

# Pivot table: TQ by ngram_range x max_features (averaged across subjects & min_docs)
print(f"\n{'='*80}")
print(f"TOPIC QUALITY HEATMAP (ngram_range x max_features, avg across subjects)")
print(f"{'='*80}")
pivot = tuning_df.pivot_table(
    index="ngram_range", columns="max_features",
    values="topic_quality", aggfunc="mean"
)
print(pivot.round(4))


Saved: ../../../../results/topicGpt/tunning/cs/tuning_results.csv
Saved: ../../../../results/topicGpt/tunning/math/tuning_results.csv
Saved: ../../../../results/topicGpt/tunning/physics/tuning_results.csv

Combined results saved: ../../../../results/topicGpt/tunning/tuning_results_all.csv

BEST CONFIGURATION PER SUBJECT

CS:
  Model:        all_mpnet_base_v2
  min_docs:     100
  ngram_range:  (1, 1)
  max_features: 10000.0
  Topics:       276
  C_v:          0.6953
  IRBO:         0.9903
  TQ:           0.8170

MATH:
  Model:        sentence_transformers_all_MiniLM_L6_v2
  min_docs:     200
  ngram_range:  (1, 1)
  max_features: 10000.0
  Topics:       124
  C_v:          0.6924
  IRBO:         0.9817
  TQ:           0.8120

PHYSICS:
  Model:        all_mpnet_base_v2
  min_docs:     150
  ngram_range:  (1, 1)
  max_features: 10000.0
  Topics:       188
  C_v:          0.7046
  IRBO:         0.9870
  TQ:           0.8222

TOPIC QUALITY HEATMAP (ngram_range x max_features, avg across su

## Save Best Tuning Model

For each subject, save the best `(min_docs, ngram_range, max_features)` assignment + config
as a checkpoint and export the final assignment CSV.

In [10]:
for subject in LIST_SUBJECT:
    if subject not in all_base_assignments:
        continue

    subj_df = tuning_df[tuning_df["subject"] == subject]
    if len(subj_df) == 0:
        continue

    best = subj_df.loc[subj_df["topic_quality"].idxmax()]
    best_min_docs    = int(best["min_docs"])
    best_ngram       = eval(best["ngram_range"]) if isinstance(best["ngram_range"], str) else best["ngram_range"]
    best_max_features = int(best["max_features"])
    best_model       = best["best_model"]

    print(f"\n{'='*60}")
    print(f"SAVING BEST FOR: {subject.upper()}")
    print(f"  model={best_model}, min_docs={best_min_docs}, ngram={best_ngram}, max_feat={best_max_features}")
    print(f"{'='*60}")

    # Re-apply filtering with best params
    base_df = all_base_assignments[subject]
    df_best = filter_and_reindex_topics(base_df, min_docs=best_min_docs)

    # Recompute metrics for verification
    texts = all_data[subject]["text"].fillna("").tolist()
    metrics = compute_coherence_irbo(
        df_best, texts,
        ngram_range=best_ngram,
        max_features=best_max_features
    )

    print(f"  Verified: C_v={metrics['coherence']:.4f}  "
          f"IRBO={metrics['irbo']:.4f}  TQ={metrics['topic_quality']:.4f}")
    print(f"  Topics: {df_best['topic_id'].nunique()}  Docs: {len(df_best)}")

    # Save best model checkpoint
    save_checkpoint(
        {
            "assignment_df": df_best,
            "metrics": metrics,
            "config": {
                "best_model":    best_model,
                "min_docs":      best_min_docs,
                "ngram_range":   best_ngram,
                "max_features":  best_max_features,
            }
        },
        "tuning_best", subject
    )

    # Export assignment CSV
    df_export = df_best.copy()
    df_export["subject"]      = subject
    df_export["best_model"]   = best_model
    df_export["min_docs"]     = best_min_docs
    df_export["ngram_range"]  = str(best_ngram)
    df_export["max_features"] = best_max_features
    df_export["coherence"]    = metrics["coherence"]
    df_export["irbo"]         = metrics["irbo"]

    out_csv = RESULT_DIR / subject / "topicgpt_assignments.csv"
    df_export.to_csv(out_csv, index=False)
    print(f"  Saved CSV: {out_csv}")
    print(f"    Columns: {list(df_export.columns)}")
    print(f"    Rows: {len(df_export)}")



SAVING BEST FOR: CS
  model=all_mpnet_base_v2, min_docs=100, ngram=(1, 1), max_feat=10000
  Verified: C_v=0.6953  IRBO=0.9903  TQ=0.8170
  Topics: 276  Docs: 88115
  Checkpoint saved: ../../../../models/topicGpt/cs/tuning_best.pkl
  Saved CSV: ../../../../results/topicGpt/tunning/cs/topicgpt_assignments.csv
    Columns: ['doc_idx', 'topic_id', 'topic_label', 'confidence', 'original_topic_id', 'subject', 'best_model', 'min_docs', 'ngram_range', 'max_features', 'coherence', 'irbo']
    Rows: 88115

SAVING BEST FOR: MATH
  model=sentence_transformers_all_MiniLM_L6_v2, min_docs=200, ngram=(1, 1), max_feat=10000
  Verified: C_v=0.6924  IRBO=0.9817  TQ=0.8120
  Topics: 124  Docs: 46626
  Checkpoint saved: ../../../../models/topicGpt/math/tuning_best.pkl
  Saved CSV: ../../../../results/topicGpt/tunning/math/topicgpt_assignments.csv
    Columns: ['doc_idx', 'topic_id', 'topic_label', 'confidence', 'original_topic_id', 'subject', 'best_model', 'min_docs', 'ngram_range', 'max_features', 'coher

## Final Summary

In [11]:
print(f"\n{'='*100}")
print(f"{'SUBJECT':<12} | {'MODEL':<45} | {'min_docs':>8} | {'ngram':>8} | {'max_feat':>9} | {'TOPICS':>6} | {'TQ':>6}")
print(f"{'-'*100}")

for subject in LIST_SUBJECT:
    subj_df = tuning_df[tuning_df["subject"] == subject]
    if len(subj_df) == 0:
        continue
    best = subj_df.loc[subj_df["topic_quality"].idxmax()]
    feat_str = str(int(best["max_features"])) if best["max_features"] is not None else "None"
    print(
        f"{subject:<12} | {best['best_model']:<45} | {int(best['min_docs']):>8} | "
        f"{best['ngram_range']:>8} | {feat_str:>9} | "
        f"{int(best['n_topics']):>6} | {best['topic_quality']:>6.4f}"
    )

print(f"\nDone! All results saved to {RESULT_DIR.resolve()}")
print(f"Best model checkpoints saved to {CHECKPOINT_DIR.resolve()}/{{subject}}/tuning_best.pkl")



SUBJECT      | MODEL                                         | min_docs |    ngram |  max_feat | TOPICS |     TQ
----------------------------------------------------------------------------------------------------
cs           | all_mpnet_base_v2                             |      100 |   (1, 1) |     10000 |    276 | 0.8170
math         | sentence_transformers_all_MiniLM_L6_v2        |      200 |   (1, 1) |     10000 |    124 | 0.8120
physics      | all_mpnet_base_v2                             |      150 |   (1, 1) |     10000 |    188 | 0.8222

Done! All results saved to /home/nedo/Kuliah/TA/Program/results/topicGpt/tunning
Best model checkpoints saved to /home/nedo/Kuliah/TA/Program/models/topicGpt/{subject}/tuning_best.pkl
